# CITYKIN WO2a — does basin relief-range measure terrain, or basin size?

Diagnostic probe ahead of WO2's Terrain regime design (coarse basin-scale lens on the sandbox
Similarity panel). Settles one facet-choice question: `relief_range` (`ele_mt_smx - ele_mt_smn`) is
built from order statistics over a basin's pixels, so a larger basin has a higher expected max and
lower expected min *before any difference in terrain* — plus a real longitudinal gradient a small
basin can't accumulate. Whether that confound is large enough to matter is a magnitude question,
measured here, not presumed.

No wiring, no persisted artifact, no UI. WO: `docs/cdop/citykin/wo2a_basin-relief.md`. Tracker:
`docs/cdop/citykin/CITYKIN_tracker.md`. Karl runs every cell and reports output back; no number here
is a finding until he's seen it.

In [1]:
# Cell 1
%matplotlib inline
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

from scripts.shared.db_utils import db_connect
import scripts.shared.db_utils as db_utils

ROOT = Path(db_utils.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'cdop' / 'citykin'
OUT.mkdir(parents=True, exist_ok=True)

print(f"ROOT: {ROOT}\nOUT : {OUT}")

ROOT: /Users/karlg/Documents/repos/_edops
OUT : /Users/karlg/Documents/repos/_edops/output/cdop/citykin


## Part A — the sample

A corpus-wide read, not a fixture list: all L06 basins' `ele_mt_sav`/`ele_mt_smn`/`ele_mt_smx`,
`slp_dg_sav`, and area, with `relief_range` and `log10(area)` derived.

In [2]:
# Cell 2 -- Part A: pull all L06 basins. Area column is `sub_area` -- confirmed against existing SQL
# in the codebase (engine.py, hyde.py both already query basin06/08.sub_area directly) before writing
# it into this cell, per the standing inspect-before-hardcoding rule; not assumed from the catalog.
# -9999 is BasinATLAS NoData (assigned to all of Greenland for slp_dg_*/sgr_dk_*) -- masked to NaN,
# never coerced to zero (CLAUDE.md). Slope is stored in degrees x10 (WO2a Part A proviso) -- converted
# on read. `_sav`/`_smn`/`_smx` only, never `_uav` (upstream-watershed aggregate, wrong facet here).
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
conn = db_connect()

l06 = pd.read_sql("""
    SELECT hybas_id, ele_mt_sav, ele_mt_smn, ele_mt_smx, slp_dg_sav, sub_area
    FROM basin06
""", conn)
conn.close()

for col in ['ele_mt_sav', 'ele_mt_smn', 'ele_mt_smx', 'slp_dg_sav', 'sub_area']:
    l06[col] = l06[col].replace(-9999, np.nan)
l06['slp_dg_sav'] = l06['slp_dg_sav'] / 10.0

l06['relief_range'] = l06['ele_mt_smx'] - l06['ele_mt_smn']
l06['log_area'] = np.log10(l06['sub_area'])

print("\n".join([
    f"L06 basins pulled: {len(l06)}",
    "NaN counts (NoData mask -- mostly Greenland slope/relief):",
    l06[['ele_mt_sav', 'ele_mt_smn', 'ele_mt_smx', 'slp_dg_sav', 'sub_area']].isna().sum().to_string(),
]))

L06 basins pulled: 16397
NaN counts (NoData mask -- mostly Greenland slope/relief):
ele_mt_sav      0
ele_mt_smn      0
ele_mt_smx      0
slp_dg_sav    302
sub_area        0


## Part B — the measurement

Three readings, in this order: (1) does `relief_range` track slope, or track area — the partial
correlation with `log10(area)` controlling for `slp_dg_sav` is the whole question; (2) is
`relief_range` redundant with slope; (3) the quartile grid, area × slope, diagnostic cell =
large-area/low-slope vs small-area/high-slope. Named basins after, for plausibility only.

In [3]:
# Cell 3 -- Part B readings 1-2. Reading 1: does relief_range track slope, or track area? Residualize
# both relief_range and log_area against slp_dg_sav (OLS), then correlate the residuals -- that
# residual correlation is the association with basin size that slope does NOT already explain, the
# whole question this WO asks. Reading 2: corr(relief_range, slp_dg_sav) -- are the two facets
# redundant, i.e. carrying one signal rather than being usefully discriminating.
ok = l06.dropna(subset=['relief_range', 'slp_dg_sav', 'log_area']).reset_index(drop=True)

def _resid(y, x):
    b = np.polyfit(x, y, 1)
    return y - (b[0] * x + b[1])

r_relief_slope = ok['relief_range'].corr(ok['slp_dg_sav'])
r_relief_area  = ok['relief_range'].corr(ok['log_area'])
r_slope_area   = ok['slp_dg_sav'].corr(ok['log_area'])

resid_relief = _resid(ok['relief_range'].to_numpy(), ok['slp_dg_sav'].to_numpy())
resid_area   = _resid(ok['log_area'].to_numpy(), ok['slp_dg_sav'].to_numpy())
partial_r = np.corrcoef(resid_relief, resid_area)[0, 1]

print("\n".join([
    f"L06, n={len(ok)} complete-case",
    f"corr(relief_range, slp_dg_sav)      = {r_relief_slope:.3f}   (reading 2: redundancy)",
    f"corr(relief_range, log_area)         = {r_relief_area:.3f}",
    f"corr(slp_dg_sav, log_area)           = {r_slope_area:.3f}",
    f"partial corr(relief_range, log_area | slp_dg_sav) = {partial_r:.3f}   <- reading 1, the whole question",
]))

L06, n=16095 complete-case
corr(relief_range, slp_dg_sav)      = 0.739   (reading 2: redundancy)
corr(relief_range, log_area)         = 0.322
corr(slp_dg_sav, log_area)           = 0.124
partial corr(relief_range, log_area | slp_dg_sav) = 0.344   <- reading 1, the whole question


In [4]:
# Cell 4 -- Part B reading 3: the quartile grid. Cross-tab mean relief_range and mean slp_dg_sav by
# area quartile x slope quartile. Diagnostic cell: large-area/low-slope vs small-area/high-slope, in
# meters, side by side -- the check in the form Karl asked for, not a coefficient.
ok = ok.copy()
ok['area_q']  = pd.qcut(ok['log_area'], 4, labels=['Q1 (smallest)', 'Q2', 'Q3', 'Q4 (largest)'])
ok['slope_q'] = pd.qcut(ok['slp_dg_sav'], 4, labels=['Q1 (flattest)', 'Q2', 'Q3', 'Q4 (steepest)'])

relief_grid = ok.pivot_table(index='area_q', columns='slope_q', values='relief_range', aggfunc='mean', observed=False)
slope_grid  = ok.pivot_table(index='area_q', columns='slope_q', values='slp_dg_sav', aggfunc='mean', observed=False)
n_grid      = ok.pivot_table(index='area_q', columns='slope_q', values='relief_range', aggfunc='count', observed=False)

print("L06 -- mean relief_range (m) by area quartile (rows) x slope quartile (cols):")
print(relief_grid.round(1).to_string())
print()
print("L06 -- mean slp_dg_sav (deg) by the same grid, for reference:")
print(slope_grid.round(2).to_string())
print()
print("L06 -- cell counts:")
print(n_grid.to_string())
print()
print(f"diagnostic cell -- large-area/low-slope relief_range = "
      f"{relief_grid.loc['Q4 (largest)', 'Q1 (flattest)']:.1f}m vs "
      f"small-area/high-slope = {relief_grid.loc['Q1 (smallest)', 'Q4 (steepest)']:.1f}m")

L06 -- mean relief_range (m) by area quartile (rows) x slope quartile (cols):
slope_q        Q1 (flattest)     Q2      Q3  Q4 (steepest)
area_q                                                    
Q1 (smallest)           82.8  291.0   615.3         1467.6
Q2                     198.8  456.5   934.8         1995.0
Q3                     268.6  545.4  1144.0         2288.5
Q4 (largest)           382.9  730.2  1458.2         2717.2

L06 -- mean slp_dg_sav (deg) by the same grid, for reference:
slope_q        Q1 (flattest)    Q2    Q3  Q4 (steepest)
area_q                                                 
Q1 (smallest)           0.33  1.48  3.72          11.06
Q2                      0.39  1.53  3.69          10.85
Q3                      0.41  1.52  3.70          10.67
Q4 (largest)            0.43  1.54  3.78          10.86

L06 -- cell counts:
slope_q        Q1 (flattest)    Q2    Q3  Q4 (steepest)
area_q                                                 
Q1 (smallest)           1573   928  

In [5]:
# Cell 5 -- plausibility only, NOT the finding (the corpus-wide readings above are). A handful of
# named basins spanning the grid, located by coordinate lookup; each confirmed as the intended basin
# (by its returned hybas_id / sub_area, eyeballed below) before reading anything from it, per the WO's
# own proviso. L06 only -- this is a sanity check on the mechanism, not a repeated measurement.
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
conn = db_connect()

CANDIDATES = [
    ('large lowland -- Amazon (near Manaus, Brazil)',   -3.10,  -60.02),
    ('large lowland -- Congo (central DRC)',              0.00,   18.00),
    ('small Alpine (near Innsbruck, Austria)',           47.26,   11.39),
    ('large flat plains (western Kansas, US)',           38.50, -100.50),
    ('small lowland (Netherlands polder)',                52.10,    5.10),
]
rows = []
for label, lat, lon in CANDIDATES:
    row = pd.read_sql(f"""
        SELECT hybas_id, ele_mt_sav, ele_mt_smn, ele_mt_smx, slp_dg_sav, sub_area
        FROM basin06
        WHERE ST_Contains(geom, ST_SetSRID(ST_MakePoint({lon}, {lat}), 4326))
        LIMIT 1
    """, conn)
    if len(row):
        r = row.iloc[0]
        slope = (r['slp_dg_sav'] / 10.0) if pd.notna(r['slp_dg_sav']) and r['slp_dg_sav'] != -9999 else np.nan
        rows.append({
            'label': label, 'hybas_id': int(r['hybas_id']),
            'ele_mt_sav': r['ele_mt_sav'],
            'relief_range': (r['ele_mt_smx'] - r['ele_mt_smn']) if pd.notna(r['ele_mt_smx']) else np.nan,
            'slp_dg_sav': slope, 'sub_area_km2': r['sub_area'],
        })
    else:
        rows.append({'label': label, 'hybas_id': None})
conn.close()

named = pd.DataFrame(rows)
print("named basins (plausibility check only -- eyeball each hybas_id/sub_area against the label "
      "before trusting the row):")
print(named.to_string(index=False))

named basins (plausibility check only -- eyeball each hybas_id/sub_area against the label before trusting the row):
                                        label   hybas_id  ele_mt_sav  relief_range  slp_dg_sav  sub_area_km2
large lowland -- Amazon (near Manaus, Brazil) 6060280400        59.0         145.0         2.4        3485.2
         large lowland -- Congo (central DRC) 1061156950       318.0          48.0         0.3        4383.7
       small Alpine (near Innsbruck, Austria) 2060465720      1251.0        3590.0        16.2       25920.1
       large flat plains (western Kansas, US) 7060622710       522.0         642.0         0.5       33459.7
           small lowland (Netherlands polder) 2060023020         3.0         121.0         0.3        6727.0


## Part C — L08 secondary

Not a separate question — a check on whether the Part B answer is scale-conditional. L08's area
spread is narrower than L06's, so the confound should be *weaker* here if area really is the
mechanism. If it is instead stronger or unchanged, the mechanism is not area and the diagnosis is
wrong — that gets reported as such, not reconciled toward the L06 reading.

In [6]:
# Cell 6 -- Part C: pull L08, same query shape and masking/scaling as Cell 2 (basin08 is the same
# verbatim BasinATLAS structure as basin06, per CLAUDE.md). L08's finer basins turned up a case L06
# didn't have: some sub_area values are <= 0 (divide-by-zero into log10, caught rather than suppressed)
# -- not the -9999 NoData sentinel, a separate zero/negative-area edge case at this finer resolution.
# Masked to NaN before log10, same discipline as -9999, with the count reported so it's visible, not
# silently dropped.
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
conn = db_connect()
l08 = pd.read_sql("""
    SELECT hybas_id, ele_mt_sav, ele_mt_smn, ele_mt_smx, slp_dg_sav, sub_area
    FROM basin08
""", conn)
conn.close()

for col in ['ele_mt_sav', 'ele_mt_smn', 'ele_mt_smx', 'slp_dg_sav', 'sub_area']:
    l08[col] = l08[col].replace(-9999, np.nan)
l08['slp_dg_sav'] = l08['slp_dg_sav'] / 10.0

n_nonpos_area = int((l08['sub_area'] <= 0).sum())
l08.loc[l08['sub_area'] <= 0, 'sub_area'] = np.nan

l08['relief_range'] = l08['ele_mt_smx'] - l08['ele_mt_smn']
l08['log_area'] = np.log10(l08['sub_area'])

print("\n".join([
    f"L08 basins pulled: {len(l08)}",
    l08[['ele_mt_sav', 'ele_mt_smn', 'ele_mt_smx', 'slp_dg_sav', 'sub_area']].isna().sum().to_string(),
    f"sub_area <= 0 (masked to NaN before log10, not counted above): {n_nonpos_area}",
]))

L08 basins pulled: 190675
ele_mt_sav       0
ele_mt_smn       0
ele_mt_smx       0
slp_dg_sav    6390
sub_area         6
sub_area <= 0 (masked to NaN before log10, not counted above): 6


In [7]:
# Cell 7 -- L08 readings 1-2 (Part C repeat of Cell 3). Same shape, same question: is the L06 reading
# scale-conditional? L08's area spread is narrower, so the confound should be *weaker* here if area is
# really the mechanism -- reported either way, not reconciled toward the L06 answer.
ok8 = l08.dropna(subset=['relief_range', 'slp_dg_sav', 'log_area']).reset_index(drop=True)

r_relief_slope8 = ok8['relief_range'].corr(ok8['slp_dg_sav'])
r_relief_area8  = ok8['relief_range'].corr(ok8['log_area'])
r_slope_area8   = ok8['slp_dg_sav'].corr(ok8['log_area'])

resid_relief8 = _resid(ok8['relief_range'].to_numpy(), ok8['slp_dg_sav'].to_numpy())
resid_area8   = _resid(ok8['log_area'].to_numpy(),   ok8['slp_dg_sav'].to_numpy())
partial_r8 = np.corrcoef(resid_relief8, resid_area8)[0, 1]

print("\n".join([
    f"L08, n={len(ok8)} complete-case",
    f"corr(relief_range, slp_dg_sav)      = {r_relief_slope8:.3f}   (reading 2: redundancy)",
    f"corr(relief_range, log_area)         = {r_relief_area8:.3f}",
    f"corr(slp_dg_sav, log_area)           = {r_slope_area8:.3f}",
    f"partial corr(relief_range, log_area | slp_dg_sav) = {partial_r8:.3f}   <- reading 1, the whole question",
    "",
    f"L06 partial corr (Cell 3, for comparison): {partial_r:.3f}",
]))

L08, n=184285 complete-case
corr(relief_range, slp_dg_sav)      = 0.809   (reading 2: redundancy)
corr(relief_range, log_area)         = 0.260
corr(slp_dg_sav, log_area)           = 0.090
partial corr(relief_range, log_area | slp_dg_sav) = 0.320   <- reading 1, the whole question

L06 partial corr (Cell 3, for comparison): 0.344


In [8]:
# Cell 8 -- L08 quartile grid (Part C repeat of Cell 4). If the confound is area-driven, L08's
# narrower area spread should make the large-area/low-slope vs small-area/high-slope gap *smaller*
# here than at L06, not larger or unchanged -- reported as-is either way, not reconciled to match.
ok8 = ok8.copy()
ok8['area_q']  = pd.qcut(ok8['log_area'], 4, labels=['Q1 (smallest)','Q2','Q3','Q4 (largest)'])
ok8['slope_q'] = pd.qcut(ok8['slp_dg_sav'], 4, labels=['Q1 (flattest)','Q2','Q3','Q4 (steepest)'])

relief_grid8 = ok8.pivot_table(index='area_q', columns='slope_q', values='relief_range', aggfunc='mean', observed=False)
slope_grid8  = ok8.pivot_table(index='area_q', columns='slope_q', values='slp_dg_sav', aggfunc='mean', observed=False)
n_grid8      = ok8.pivot_table(index='area_q', columns='slope_q', values='relief_range', aggfunc='count', observed=False)

diag8 = relief_grid8.loc['Q4 (largest)', 'Q1 (flattest)'] - relief_grid8.loc['Q1 (smallest)', 'Q4 (steepest)']
diag6 = relief_grid.loc['Q4 (largest)', 'Q1 (flattest)'] - relief_grid.loc['Q1 (smallest)', 'Q4 (steepest)']

print("L08 -- mean relief_range (m) by area quartile (rows) x slope quartile (cols):")
print(relief_grid8.round(1).to_string())
print()
print("L08 -- mean slp_dg_sav (deg) by the same grid, for reference:")
print(slope_grid8.round(2).to_string())
print()
print("L08 -- cell counts:")
print(n_grid8.to_string())
print()
print(f"L08 diagnostic gap (large-flat minus small-rugged): {diag8:.1f}m")
print(f"L06 diagnostic gap (Cell 4, for comparison):        {diag6:.1f}m")
print(f"L08 gap {'smaller' if abs(diag8) < abs(diag6) else 'NOT smaller'} than L06 "
      f"-- {'consistent with an area mechanism' if abs(diag8) < abs(diag6) else 'inconsistent with an area mechanism; report, do not reconcile'}")

L08 -- mean relief_range (m) by area quartile (rows) x slope quartile (cols):
slope_q        Q1 (flattest)     Q2     Q3  Q4 (steepest)
area_q                                                   
Q1 (smallest)           38.5  123.7  299.4          959.4
Q2                      79.7  201.5  439.4         1286.9
Q3                     105.6  250.2  546.2         1484.6
Q4 (largest)           143.2  327.3  706.2         1754.5

L08 -- mean slp_dg_sav (deg) by the same grid, for reference:
slope_q        Q1 (flattest)    Q2    Q3  Q4 (steepest)
area_q                                                 
Q1 (smallest)           0.27  1.30  3.45          11.99
Q2                      0.31  1.33  3.46          12.03
Q3                      0.32  1.33  3.46          11.86
Q4 (largest)            0.33  1.33  3.48          11.50

L08 -- cell counts:
slope_q        Q1 (flattest)     Q2     Q3  Q4 (steepest)
area_q                                                   
Q1 (smallest)          16286  11123   

## Next

Decision rule is stated in the WO before these numbers (`docs/cdop/citykin/wo2a_basin-relief.md` §
Decision rule), not fit to them after the fact. Karl sets the "weak"/"substantial" line after seeing
the spread above, in that order. Once set, the facet choice, the rejected one, and the numbers that
decided between them go into `wo2a_findings.md`, and the choice into `CITYKIN_tracker.md` § Locked
decisions with its numeric basis, in the same edit (validation order step 4). No wiring, no persisted
column, no UI in this WO.

## WO2b — the shipping-facet correlation, and what Kansas is evidence of

Closing diagnostic (`docs/cdop/citykin/wo2b_followup.md`), Opus's follow-up on WO2a. Two gaps: (1)
WO2a's redundancy reading was `corr(relief_range, slp_dg_sav)` — but `slp_dg_sav` isn't a WO2 shipping
facet; the correlation the lens actually depends on, `corr(ele_mt_sav, relief_range)`, was never
reported. (2) `wo2a_findings.md` cited the Kansas basin for two different conclusions (confound
illustration and keep-relief illustration) without separating them empirically. Also noted: WO2a's
decision rule imported the project's 0.70 Mahalanobis/drop bar without asking what it's for — that bar
exists to correct double-counting in a compensatory (quadrature-sum) distance; WO2's lens is a
non-compensatory tolerance-band conjunction, where correlated facets cost *selectivity*, not distance
integrity. Neither reading below is a gate — see the WO's own decision rule. No wiring, no third
facet, no UI.

In [9]:
# Cell 9 -- WO2b Part A: the shipping-facet correlation. `ok`/`ok8` already carry `ele_mt_sav` and
# `relief_range` from Cells 2/6 -- no re-derivation, this reads the frame WO2a already built. Reported
# as a description (with its scatter, so the shape is visible, not only the coefficient), not compared
# to the 0.70 bar -- see the WO2b decision rule (this lens is non-compensatory; correlated facets cost
# selectivity, not distance integrity).
r_ship_l06 = ok['ele_mt_sav'].corr(ok['relief_range'])
r_ship_l08 = ok8['ele_mt_sav'].corr(ok8['relief_range'])

print("drawing WO2b Part A scatter (ele_mt_sav vs relief_range, L06 + L08)...")
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, df, label, r in [(axes[0], ok, 'L06', r_ship_l06), (axes[1], ok8, 'L08', r_ship_l08)]:
    ax.set_facecolor('white')
    ax.hexbin(df['ele_mt_sav'], df['relief_range'], gridsize=60, bins='log', mincnt=1)
    ax.set_xlabel('ele_mt_sav (m)', color='black')
    ax.set_ylabel('relief_range (m)', color='black')
    ax.set_title(f"{label}: corr = {r:.3f}", color='black')
    ax.tick_params(colors='black')
fig.patch.set_facecolor('white')
fig.tight_layout()
out_path = OUT / 'wo2b_shipping_facet_scatter.png'
fig.savefig(out_path, facecolor='white')
plt.close(fig)
display(IPImage(str(out_path)))

print(f"corr(ele_mt_sav, relief_range): L06 = {r_ship_l06:.3f}   L08 = {r_ship_l08:.3f}")
print("(reading only -- not compared to the 0.70 bar; see the WO2b decision rule)")

corr(ele_mt_sav, relief_range): L06 = 0.541   L08 = 0.566
(reading only -- not compared to the 0.70 bar; see the WO2b decision rule)


In [10]:
# Cell 10 -- WO2b Part B: what Kansas is evidence of. The western-Kansas basin (hybas_id
# 7060622710) is already a row in `ok` (Cell 4 already assigned it area_q/slope_q) -- no re-query.
# Two conditional expectations, separated: (1) expected relief given slope alone, corpus-wide
# (marginal over area -- the reading that would make it a keep-relief illustration); (2) expected
# relief given area AND slope quartile jointly -- literally the Cell 4 grid cell it falls into (the
# reading that would make it a pure-confound illustration). If Kansas's actual relief exceeds (1) but
# sits near (2), its excess is size, and it belongs only in the confound paragraph; if it exceeds both,
# it is genuinely locally distinctive.
KANSAS_HYBAS = 7060622710
kansas = ok[ok['hybas_id'] == KANSAS_HYBAS].iloc[0]

exp_given_slope = ok[ok['slope_q'] == kansas['slope_q']]['relief_range'].mean()
exp_given_area_and_slope = relief_grid.loc[kansas['area_q'], kansas['slope_q']]

print("\n".join([
    f"Kansas basin {KANSAS_HYBAS}: actual relief_range = {kansas['relief_range']:.1f}m "
    f"(area_q={kansas['area_q']}, slope_q={kansas['slope_q']}, "
    f"sub_area={kansas['sub_area']:.0f}km2, slp_dg_sav={kansas['slp_dg_sav']:.2f} deg)",
    "",
    f"reading 1 -- expected relief given slope alone, corpus-wide (marginal over area): "
    f"{exp_given_slope:.1f}m  (Kansas exceeds by {kansas['relief_range'] - exp_given_slope:.1f}m)",
    f"reading 2 -- expected relief given area+slope quartile jointly (Cell 4 grid cell): "
    f"{exp_given_area_and_slope:.1f}m  (Kansas exceeds by "
    f"{kansas['relief_range'] - exp_given_area_and_slope:.1f}m)",
]))

Kansas basin 7060622710: actual relief_range = 642.0m (area_q=Q4 (largest), slope_q=Q1 (flattest), sub_area=33460km2, slp_dg_sav=0.50 deg)

reading 1 -- expected relief given slope alone, corpus-wide (marginal over area): 206.9m  (Kansas exceeds by 435.1m)
reading 2 -- expected relief given area+slope quartile jointly (Cell 4 grid cell): 382.9m  (Kansas exceeds by 259.1m)


## WO3 Part B — deriving the Terrain regime lens's tolerance-band defaults

`docs/cdop/citykin/wo3_coarse-terrain.md` Part B. WO3 Part A (`app/db/seasonality.py`) built and
live-verified the tolerance core with placeholder band defaults borrowed from WO1a's point-window
lens (elevation ±500m, relief ±300m) — a different instrument over a different corpus, kept only so
the code had *some* default while this part derives the real ones.

Two things to derive, in order: (1) three round-fraction levels (tight/default/broad) per facet from
this corpus's own spread — the same pattern WO1a used; (2) whether the *joint* admission rate at the
candidate default runs looser than independence would predict, per WO2b's proviso
(`corr(ele_mt_sav, relief_range)` = 0.54/0.57 means the second band frequently admits what the first
already did). Query basins are sampled by elevation-quartile × relief-quartile from the table, not
coordinate-picked — the standing lesson from WO2a's Innsbruck miss, now also named directly in WO3
Part C's proviso.

No number here is a finding until Karl has seen it and the levels are confirmed before they replace
the placeholders in `seasonality.py`.

In [11]:
# Cell 11 -- corpus spread of the two shipping facets, full L06 (16,397 basins, 0 NaN for either --
# Cell 2's NaN report). Reusing `l06`, not the slope-filtered `ok` -- the terrain lens does not depend
# on slp_dg_sav, so there is no reason to drop Greenland's 302 basins from this corpus's own spread.
elev_std  = l06['ele_mt_sav'].std()
relief_std = l06['relief_range'].std()

desc = l06[['ele_mt_sav', 'relief_range']].describe().round(1)
print("L06 descriptive stats, full corpus (n=16,397):")
print(desc.to_string())
print()
print(f"ele_mt_sav std:    {elev_std:.1f}m")
print(f"relief_range std:  {relief_std:.1f}m")

L06 descriptive stats, full corpus (n=16,397):
       ele_mt_sav  relief_range
count     16397.0       16397.0
mean        613.4        1033.5
std         775.3        1155.4
min         -30.0           0.0
25%         157.0         228.0
50%         364.0         593.0
75%         774.0        1510.0
max        5556.0        8530.0

ele_mt_sav std:    775.3m
relief_range std:  1155.4m


In [12]:
# Cell 12 -- candidate three-level bands as round fractions of each facet's own std (WO1a's pattern:
# tight/default/broad at roughly half/three-quarter/one full std), rounded to a plain 50m step so the
# knobs read as round numbers, not fitted decimals. Candidates only -- Cell 14 checks whether the
# resulting joint (not just per-facet) admission rate at the default level actually reads as selective
# before these replace the seasonality.py placeholders.
def round_frac(std, frac, base=50.0):
    return round(std * frac / base) * base

LEVELS = [('tight', 0.5), ('default', 0.75), ('broad', 1.0)]
ELEV_LEVELS   = {name: round_frac(elev_std, frac)   for name, frac in LEVELS}
RELIEF_LEVELS = {name: round_frac(relief_std, frac) for name, frac in LEVELS}

print(f"terrain_elev candidate levels (elev std={elev_std:.1f}m):   {ELEV_LEVELS}")
print(f"terrain_relief candidate levels (relief std={relief_std:.1f}m): {RELIEF_LEVELS}")

terrain_elev candidate levels (elev std=775.3m):   {'tight': 400.0, 'default': 600.0, 'broad': 800.0}
terrain_relief candidate levels (relief std=1155.4m): {'tight': 600.0, 'default': 850.0, 'broad': 1150.0}


In [13]:
# Cell 13 -- sample query basins by elevation-quartile x relief-quartile from the table (seeded
# random pick per cell), NOT coordinate-picked -- the standing lesson from WO2a's Innsbruck miss,
# named directly as a WO3 Part C proviso and applied here too since Part B needs "a spread of query
# basins," not named places.
rng = np.random.default_rng(42)
l06['elev_q']   = pd.qcut(l06['ele_mt_sav'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
l06['relief_q'] = pd.qcut(l06['relief_range'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

sample_rows = []
for eq in ['Q1', 'Q2', 'Q3', 'Q4']:
    for rq in ['Q1', 'Q2', 'Q3', 'Q4']:
        cell = l06[(l06['elev_q'] == eq) & (l06['relief_q'] == rq)]
        if len(cell):
            pick = cell.sample(1, random_state=int(rng.integers(1_000_000)))
            sample_rows.append(pick.iloc[0])

sample_df = pd.DataFrame(sample_rows).reset_index(drop=True)
print(f"sampled {len(sample_df)} query basins across the 4x4 elevation x relief quartile grid "
      f"(seeded rng=42, not coordinate-picked):")
print(sample_df[['hybas_id', 'ele_mt_sav', 'relief_range', 'elev_q', 'relief_q']]
      .round(1).to_string(index=False))

sampled 16 query basins across the 4x4 elevation x relief quartile grid (seeded rng=42, not coordinate-picked):
    hybas_id  ele_mt_sav  relief_range elev_q relief_q
5060472480.0          36            27     Q1       Q1
5060164140.0          62           401     Q1       Q2
5060176860.0          73          1171     Q1       Q3
5060041500.0          59          2289     Q1       Q4
4060209040.0         179           187     Q2       Q1
1060053710.0         356           334     Q2       Q2
3060014650.0         203          1393     Q2       Q3
8060003730.0         183          2263     Q2       Q4
8060202910.0         439           157     Q3       Q1
6060502730.0         516           467     Q3       Q2
7060622710.0         522           642     Q3       Q3
1060042200.0         528          2869     Q3       Q4
1061601190.0         979           191     Q4       Q1
1061420060.0        1089           248     Q4       Q2
4060150760.0        1309          1391     Q4       Q3
40600506

In [14]:
# Cell 14 -- WO2b's proviso: is the joint admission rate looser than independence predicts? Runs the
# REAL WO3 Part A code (app.db.seasonality.find_conjunction), not a notebook reimplementation -- this
# doubles as a second check on Part A's implementation. `per_condition` (each band alone) x
# independence gives the predicted joint count if the two bands worked independently; `set_size` is
# what the non-compensatory AND actually returns. The gap between them is the correlation's cost in
# selectivity (WO2b), measured directly rather than inferred from corr=0.54 alone.
from app.db.seasonality import load_similarity_index, find_conjunction

conn = db_connect()
load_similarity_index(conn, level=6)
conn.close()

default_bands = {'terrain_elev': ELEV_LEVELS['default'], 'terrain_relief': RELIEF_LEVELS['default']}
N = len(l06)

rows = []
for _, r in sample_df.iterrows():
    meta, _ = find_conjunction(int(r['hybas_id']), lens_id='terrain.regime', bands=default_bands, level=6)
    pc = meta['per_condition']
    indep_pred = pc['terrain_elev'] * pc['terrain_relief'] / N
    rows.append({
        'hybas_id': int(r['hybas_id']), 'elev_q': r['elev_q'], 'relief_q': r['relief_q'],
        'marginal_elev_n': pc['terrain_elev'], 'marginal_relief_n': pc['terrain_relief'],
        'joint_n_actual': meta['set_size'],
        'joint_n_indep_predicted': round(indep_pred, 1),
        'pct_corpus_actual': round(100 * meta['set_size'] / N, 3),
    })

result = pd.DataFrame(rows)
ratio = (result['joint_n_actual'] / result['joint_n_indep_predicted'].replace(0, np.nan)).mean()

print(f"terrain.regime @ candidate default bands {default_bands}, n={N} corpus:")
print(result.to_string(index=False))
print()
print(f"mean actual joint set size:        {result['joint_n_actual'].mean():.1f} "
      f"({result['pct_corpus_actual'].mean():.3f}% of corpus)")
print(f"mean independence-predicted size:  {result['joint_n_indep_predicted'].mean():.1f}")
print(f"actual / independence-predicted:   {ratio:.2f}x  <- WO2b's proviso, measured directly")

terrain.regime @ candidate default bands {'terrain_elev': 600.0, 'terrain_relief': 850.0}, n=16397 corpus:
  hybas_id elev_q relief_q  marginal_elev_n  marginal_relief_n  joint_n_actual  joint_n_indep_predicted  pct_corpus_actual
5060472480     Q1       Q1            11434               9964            8673                   6948.1             52.894
5060164140     Q1       Q2            11635              11489            9812                   8152.4             59.840
5060176860     Q1       Q3            11705               8422            5996                   6012.0             36.568
5060041500     Q1       Q4            11615               3366            1320                   2384.3              8.050
4060209040     Q2       Q1            12326              10679            9564                   8027.6             58.328
1060053710     Q2       Q2            13177              11242           10335                   9034.3             63.030
3060014650     Q2       Q3      

In [15]:
# Cell 15 -- sweep all three candidate levels across the same sample, so the shipped default can be
# judged against a spread rather than one number. WO3 Part B: a basin-scale set of a fraction of a
# percent reads as properly selective (unlike the WH Cities corpus's 9-14% -- do not import that
# caution here). This is the check for whether 'default' actually lands there.
sweep_rows = []
for level_name in ['tight', 'default', 'broad']:
    bands = {'terrain_elev': ELEV_LEVELS[level_name], 'terrain_relief': RELIEF_LEVELS[level_name]}
    sizes = []
    for _, r in sample_df.iterrows():
        meta, _ = find_conjunction(int(r['hybas_id']), lens_id='terrain.regime', bands=bands, level=6)
        sizes.append(meta['set_size'])
    sweep_rows.append({
        'level': level_name, 'elev_band': bands['terrain_elev'], 'relief_band': bands['terrain_relief'],
        'mean_set_size': round(float(np.mean(sizes)), 1),
        'median_set_size': float(np.median(sizes)),
        'max_set_size': int(np.max(sizes)),
        'mean_pct_corpus': round(100 * float(np.mean(sizes)) / N, 3),
    })

sweep = pd.DataFrame(sweep_rows)
print(f"terrain.regime set size across the sample (n={len(sample_df)} query basins), by knob level:")
print(sweep.to_string(index=False))

terrain.regime set size across the sample (n=16 query basins), by knob level:
  level  elev_band  relief_band  mean_set_size  median_set_size  max_set_size  mean_pct_corpus
  tight      400.0        600.0         4022.4           2521.0          9182           24.532
default      600.0        850.0         5910.8           5277.0         11519           36.048
  broad      800.0       1150.0         7939.1           9373.0         12683           48.418


In [16]:
# Cell 16 -- Karl flagged the negative ele_mt_sav min (-30.0m, Cell 11). Checking whether this is the
# same failure mode as WO1a's OpenTopoData bathymetric-contamination bug (wo1a_findings.md) or a
# genuine below-sea-level basin, rather than assuming either way. Key structural difference up front:
# that bug came from live point-window grid sampling landing in open water (depths to -1249m);
# ele_mt_sav is BasinATLAS's own precomputed basin statistic from a DEM over the actual watershed
# polygon, a different pipeline -- and -30m is a much milder extreme than -1249m. Checked, not assumed.
neg = l06[l06['ele_mt_sav'] < 0].copy()
print(f"basins with ele_mt_sav < 0: {len(neg)} of {len(l06)} ({100*len(neg)/len(l06):.2f}%)")
print()
print("full distribution of negative values:")
print(neg['ele_mt_sav'].describe().round(1).to_string())
print()

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
conn = db_connect()
neg_hy = tuple(int(x) for x in neg['hybas_id'])
loc = pd.read_sql(f"""
    SELECT hybas_id, ST_Y(ST_PointOnSurface(geom)) AS lat, ST_X(ST_PointOnSurface(geom)) AS lon
    FROM basin06 WHERE hybas_id IN {neg_hy}
""", conn)
conn.close()
loc['hybas_id'] = loc['hybas_id'].astype(np.int64)
neg = neg.merge(loc, on='hybas_id', how='left')

print("negative-elevation basins, located (eyeball against known below-sea-level regions -- Caspian "
      "lowlands, Netherlands, Dead Sea rift, Death Valley, Qattara Depression -- before concluding):")
print(neg[['hybas_id', 'ele_mt_sav', 'ele_mt_smn', 'ele_mt_smx', 'relief_range', 'lat', 'lon']]
      .sort_values('ele_mt_sav').round(1).to_string(index=False))

basins with ele_mt_sav < 0: 70 of 16397 (0.43%)

full distribution of negative values:
count    70.0
mean    -16.1
std       9.2
min     -30.0
25%     -23.8
50%     -17.0
75%      -9.2
max      -1.0

negative-elevation basins, located (eyeball against known below-sea-level regions -- Caspian lowlands, Netherlands, Dead Sea rift, Death Valley, Qattara Depression -- before concluding):
    hybas_id  ele_mt_sav  ele_mt_smn  ele_mt_smx  relief_range   lat   lon
2060618760.0         -30         -30         -30             0  41.2  53.4
2060619050.0         -30         -30         -30             0  41.2  53.7
2060068180.0         -29         -32         -25             7  46.4  49.3
1060100010.0         -29         -34         -15            19  34.3   6.1
2060068690.0         -29         -31         -25             6  45.6  47.7
2060067660.0         -28         -32         -22            10  47.0  52.2
2060068260.0         -28         -32         -21            11  46.2  49.0
2060068820.0 

In [17]:
# Cell 17 -- Cell 15 confirmed the std-based recipe is the wrong tool here: even 'tight' (400m/600m,
# 0.5 std) admits 24.5% of the corpus on average. Elevation's std (775m) sits above its own IQR
# (157-774m = 617m, Cell 11) -- the long right tail (max 5556m) inflates std well past the density
# where most basins actually live, so a std-fraction band is much wider, in practice, than it looks
# on paper. Dropping the std anchor entirely: sweep much smaller, roughly geometric absolute widths
# directly, on the same 16-basin sample, and read off where selectivity actually lands -- the
# quartile-grid structure of the sample (Cell 13) should keep this representative, not skewed toward
# one region of the joint distribution.
ELEV_WIDTHS   = [25, 50, 75, 100, 150, 200, 300, 400]
RELIEF_WIDTHS = [50, 75, 100, 150, 200, 300, 450, 600]

grid_rows = []
for ew in ELEV_WIDTHS:
    row = {'elev_band': ew}
    for rw in RELIEF_WIDTHS:
        sizes = [find_conjunction(int(r['hybas_id']), lens_id='terrain.regime',
                                   bands={'terrain_elev': ew, 'terrain_relief': rw}, level=6)[0]['set_size']
                 for _, r in sample_df.iterrows()]
        row[rw] = round(100 * np.mean(sizes) / N, 3)
    grid_rows.append(row)

grid = pd.DataFrame(grid_rows).set_index('elev_band')
grid.columns.name = 'relief_band'
print(f"mean % of corpus (n={len(sample_df)} query basins) matched, by (elev_band, relief_band):")
print(grid.to_string())

mean % of corpus (n=16 query basins) matched, by (elev_band, relief_band):
relief_band     50     75    100    150     200     300     450     600
elev_band                                                              
25           0.370  0.523  0.674  0.948   1.181   1.601   1.998   2.240
50           0.740  1.007  1.276  1.798   2.255   3.052   3.905   4.374
75           0.983  1.367  1.742  2.475   3.124   4.243   5.524   6.192
100          1.225  1.718  2.197  3.138   3.976   5.413   7.020   7.891
150          1.684  2.411  3.109  4.474   5.664   7.658   9.885  11.178
200          2.043  2.937  3.811  5.581   7.170   9.672  12.434  14.131
300          2.724  3.936  5.101  7.461   9.616  13.132  16.971  19.449
400          3.302  4.803  6.255  9.110  11.749  16.270  21.298  24.532


In [18]:
# Cell 18 -- confirmatory check before wiring anything: Cell 17 only showed the MEAN %-of-corpus
# across the 16-basin sample. Cell 14 already showed the same nominal band can swing widely by query
# location (7-70% at the old candidates) -- so before locking a triple into seasonality.py, look at
# the full per-basin spread (min/median/max), not just the average, for the candidate read off Cell 17:
# tight = elev 25m / relief 50m, default = elev 50m / relief 100m, broad = elev 100m / relief 200m.
CANDIDATE_LEVELS = {
    'tight':   {'terrain_elev': 25.0,  'terrain_relief': 50.0},
    'default': {'terrain_elev': 50.0,  'terrain_relief': 100.0},
    'broad':   {'terrain_elev': 100.0, 'terrain_relief': 200.0},
}

detail_rows = []
for level_name, bands in CANDIDATE_LEVELS.items():
    for _, r in sample_df.iterrows():
        meta, _ = find_conjunction(int(r['hybas_id']), lens_id='terrain.regime', bands=bands, level=6)
        detail_rows.append({
            'level': level_name, 'hybas_id': int(r['hybas_id']),
            'elev_q': r['elev_q'], 'relief_q': r['relief_q'],
            'set_size': meta['set_size'], 'pct_corpus': 100 * meta['set_size'] / N,
        })
detail = pd.DataFrame(detail_rows)

summary = detail.groupby('level')['pct_corpus'].agg(['min', 'median', 'mean', 'max']).round(3)
summary = summary.reindex(['tight', 'default', 'broad'])
print(f"terrain.regime %% of corpus matched, full spread across n={len(sample_df)} query basins "
      f"(not just the mean):")
print(summary.to_string())
print()
print("worst-case (max) rows, per level -- where each candidate is at its least selective:")
for level_name in ['tight', 'default', 'broad']:
    sub = detail[detail['level'] == level_name].sort_values('pct_corpus', ascending=False)
    print(f"\n{level_name}:")
    print(sub[['hybas_id', 'elev_q', 'relief_q', 'set_size', 'pct_corpus']].head(3).round(3).to_string(index=False))

terrain.regime %% of corpus matched, full spread across n=16 query basins (not just the mean):
           min  median   mean     max
level                                
tight    0.000   0.146  0.370   2.299
default  0.030   0.445  1.276   6.324
broad    0.085   1.656  3.976  14.381

worst-case (max) rows, per level -- where each candidate is at its least selective:

tight:
  hybas_id elev_q relief_q  set_size  pct_corpus
5060472480     Q1       Q1       377       2.299
4060209040     Q2       Q1       182       1.110
5060164140     Q1       Q2       104       0.634

default:
  hybas_id elev_q relief_q  set_size  pct_corpus
5060472480     Q1       Q1      1037       6.324
4060209040     Q2       Q1       703       4.287
5060164140     Q1       Q2       405       2.470

broad:
  hybas_id elev_q relief_q  set_size  pct_corpus
4060209040     Q2       Q1      2358      14.381
5060472480     Q1       Q1      1994      12.161
1060053710     Q2       Q2      1448       8.831


## WO3 Part C — the two-fixture generalization check

`docs/cdop/citykin/wo3_coarse-terrain.md` Part C. Defaults locked into `seasonality.py` (Cell 18's
accepted triple: elev/relief 25/50m tight, 50/100m default, 100/200m broad). Standing rule from WO1a:
a rugged query and a flat query, at the *same* default knob settings, no per-fixture tuning — single-
fixture validation is exactly how the 400m gate slipped through in WO1.

**Rugged fixture**: Tbilisi's own L06 basin — explicitly sanctioned by the WO as the rugged case
without needing quantile selection (it's the project's existing canonical terrain fixture). **Flat
fixture**: selected by area-quantile x relief-quantile from the table, not coordinate-picked — the
standing lesson from WO2a's Innsbruck miss, named directly as this WO's own proviso. Also folds in the
discrimination check named in the session opener: does the rugged query's set visibly exclude the flat
basin and vice versa, or is aggregate relief too lossy to separate them even coarsely.

In [19]:
# Cell 19 -- select the two fixtures. Rugged: Tbilisi's own L06 basin, resolved by coordinate (WO's
# own sanctioned exception -- already the project's canonical terrain fixture, confirmed live in Part
# A: hybas_id 2060616700, elevation 1638m, relief_range 3583m). Flat: NOT coordinate-picked -- selected
# from the table as the largest-area basin within the flattest relief quartile (area_q4='Q4' x
# relief_q='Q1', both already-computed quartile columns from Cells 13/17), per WO3 Part C's proviso.
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
conn = db_connect()
tbilisi_row = pd.read_sql("""
    SELECT hybas_id, ele_mt_sav, ele_mt_smx, ele_mt_smn, sub_area
    FROM basin06
    WHERE ST_Contains(geom, ST_SetSRID(ST_MakePoint(44.8015, 41.6938), 4326))
    LIMIT 1
""", conn)
conn.close()
RUGGED_HYBAS = int(tbilisi_row.iloc[0]['hybas_id'])

l06['area_q4'] = pd.qcut(l06['log_area'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
flat_candidates = l06[(l06['area_q4'] == 'Q4') & (l06['relief_q'] == 'Q1')]
flat_fixture = flat_candidates.sort_values('sub_area', ascending=False).iloc[0]
FLAT_HYBAS = int(flat_fixture['hybas_id'])

rugged = l06[l06['hybas_id'] == RUGGED_HYBAS].iloc[0]
print("\n".join([
    f"RUGGED fixture (Tbilisi's L06 basin): hybas_id={RUGGED_HYBAS}, "
    f"elev={rugged['ele_mt_sav']:.0f}m, relief_range={rugged['relief_range']:.0f}m, "
    f"sub_area={rugged['sub_area']:.0f}km2",
    f"FLAT fixture (largest-area basin in the flattest relief quartile): hybas_id={FLAT_HYBAS}, "
    f"elev={flat_fixture['ele_mt_sav']:.0f}m, relief_range={flat_fixture['relief_range']:.0f}m, "
    f"sub_area={flat_fixture['sub_area']:.0f}km2",
    f"(for comparison, WO2a's coordinate-picked Kansas basin 7060622710: relief_range=642m, "
    f"sub_area=33460km2 -- {'same basin' if FLAT_HYBAS == 7060622710 else 'a different basin'})",
]))

RUGGED fixture (Tbilisi's L06 basin): hybas_id=2060616700, elev=1638m, relief_range=3583m, sub_area=23251km2
FLAT fixture (largest-area basin in the flattest relief quartile): hybas_id=6060269510, elev=123m, relief_range=199m, sub_area=78451km2
(for comparison, WO2a's coordinate-picked Kansas basin 7060622710: relief_range=642m, sub_area=33460km2 -- a different basin)


In [20]:
# Cell 20 -- run both fixtures at the locked defaults (no bands override -- this exercises exactly
# what a user gets on first load of the lens). find_conjunction's own `members` list only carries
# hybas_id/corr/pre_total_mm/lat/lon (generic across all lenses, climate-shaped) -- enrich with each
# member's own elev/relief_range by joining back to `l06`, already fully loaded, to actually check
# terrain-coherence rather than trusting set_size alone.
rugged_meta, rugged_members = find_conjunction(RUGGED_HYBAS, lens_id='terrain.regime', level=6)
flat_meta, flat_members = find_conjunction(FLAT_HYBAS, lens_id='terrain.regime', level=6)

def enrich(members):
    ids = [m['hybas_id'] for m in members]
    sub = l06[l06['hybas_id'].isin(ids)][['hybas_id', 'ele_mt_sav', 'relief_range']]
    return sub

rugged_enriched = enrich(rugged_members)
flat_enriched   = enrich(flat_members)

print(f"RUGGED query (Tbilisi, elev={rugged['ele_mt_sav']:.0f}m, relief={rugged['relief_range']:.0f}m): "
      f"set_size={rugged_meta['set_size']}")
if len(rugged_enriched):
    print(rugged_enriched.describe().round(1).to_string())
print()
print(f"FLAT query (elev={flat_fixture['ele_mt_sav']:.0f}m, relief={flat_fixture['relief_range']:.0f}m): "
      f"set_size={flat_meta['set_size']}")
if len(flat_enriched):
    print(flat_enriched.describe().round(1).to_string())

RUGGED query (Tbilisi, elev=1638m, relief=3583m): set_size=6
           hybas_id  ele_mt_sav  relief_range
count  6.000000e+00         6.0           6.0
mean   3.393935e+09      1624.5        3561.8
std    2.160137e+09        35.3          50.0
min    1.060666e+09      1588.0        3484.0
25%    2.060761e+09      1599.8        3532.8
50%    3.060602e+09      1621.0        3573.0
75%    4.060465e+09      1634.8        3602.8
max    7.060505e+09      1685.0        3609.0

FLAT query (elev=123m, relief=199m): set_size=880
           hybas_id  ele_mt_sav  relief_range
count  8.800000e+02       880.0         880.0
mean   4.053527e+09       121.2         190.6
std    2.199873e+09        28.5          56.7
min    1.060007e+09        73.0          99.0
25%    2.060299e+09        96.0         143.0
50%    4.060054e+09       122.0         187.0
75%    6.060333e+09       146.0         235.0
max    8.060271e+09       173.0         299.0


In [22]:
# Cell 21 -- discrimination check (the session-opener question folded into Part C): does the rugged
# query's set actually exclude the flat basin and vice versa, or is aggregate relief too lossy to tell
# them apart even coarsely? Also the specific leak risk flagged in this session's log: could a
# large-flat basin's area-inflated relief_range land it inside a genuinely rugged query's band despite
# being flat by elevation? At the now-locked +-50m elevation band this should be structurally screened
# out (Tbilisi 1638m vs a flat basin nowhere near that) -- checked directly, not assumed.
rugged_ids = {m['hybas_id'] for m in rugged_members}
flat_ids   = {m['hybas_id'] for m in flat_members}

print("\n".join([
    f"FLAT_HYBAS ({FLAT_HYBAS}) in RUGGED query's set: {FLAT_HYBAS in rugged_ids}",
    f"RUGGED_HYBAS ({RUGGED_HYBAS}) in FLAT query's set: {RUGGED_HYBAS in flat_ids}",
    f"overlap between the two sets: {len(rugged_ids & flat_ids)} basins "
    f"(of {len(rugged_ids)} rugged / {len(flat_ids)} flat)",
    "",
    "elevation range spanned by each set (the leak-risk check -- do the two sets' elevation "
    "ranges stay separated, or does the flat set's relief inflation let it reach into rugged range?):",
    f"  rugged set elev range: {rugged_enriched['ele_mt_sav'].min():.0f}m - "
    f"{rugged_enriched['ele_mt_sav'].max():.0f}m" if len(rugged_enriched) else "  rugged set: empty",
    f"  flat set elev range:   {flat_enriched['ele_mt_sav'].min():.0f}m - "
    f"{flat_enriched['ele_mt_sav'].max():.0f}m" if len(flat_enriched) else "  flat set: empty",
]))

FLAT_HYBAS (6060269510) in RUGGED query's set: False
RUGGED_HYBAS (2060616700) in FLAT query's set: False
overlap between the two sets: 0 basins (of 6 rugged / 880 flat)

elevation range spanned by each set (the leak-risk check -- do the two sets' elevation ranges stay separated, or does the flat set's relief inflation let it reach into rugged range?):
  rugged set elev range: 1588m - 1685m
  flat set elev range:   73m - 173m
